# Data Preparation and Cleaning

In [9]:
import requests
import pandas as pd

## Import Libraries


## Load Census API Data (2012, 2014)

In [24]:
import os
CENSUS_API_KEY = "c162f3620fb054f8b97d67a3ef165e8a8417a185"
csv_path = "../data/census_2012_2014_sba.csv"

if os.path.exists(csv_path):
    print(f"File '{csv_path}' exists. Skipping API data load.")
    raw_census_df = pd.read_csv(csv_path)
    print(f"Loaded dataframe from '{csv_path}' with {len(raw_census_df)} rows and {len(raw_census_df.columns)} columns.")
else:
    all_dataframes = []
    print("Fetching data for 2012...")
    url_2012 = "https://api.census.gov/data/2012/sbo/cs"
    vars_2012 = "NAME,NAICS2012,NAICS2012_LABEL,SEX,SEX_LABEL,ETH_GROUP,ETH_GROUP_LABEL,RACE_GROUP,RACE_GROUP_LABEL,VET_GROUP,VET_GROUP_LABEL,FIRMALL,RCPALL,EMP"
    params_2012 = {"get": vars_2012, "for": "state:*"}
    if CENSUS_API_KEY:
        params_2012["key"] = CENSUS_API_KEY
    response_2012 = requests.get(url_2012, params=params_2012)
    response_2012.raise_for_status()
    data_2012 = response_2012.json()
    df_2012 = pd.DataFrame(data_2012[1:], columns=data_2012[0])
    df_2012['YEAR'] = 2012
    all_dataframes.append(df_2012)
    print(f"Successfully fetched {len(df_2012)} rows for 2012.\n")

    print("Fetching data for 2014...")
    url_2014 = "https://api.census.gov/data/2014/ase/csa"
    vars_2014 = "NAME,NAICS2012,NAICS2012_TTL,SEX,SEX_TTL,ETH_GROUP,ETH_GROUP_TTL,RACE_GROUP,RACE_GROUP_TTL,VET_GROUP,VET_GROUP_TTL,FIRMPDEMP,RCPPDEMP,EMP"
    params_2014 = {"get": vars_2014, "for": "state:*"}
    if CENSUS_API_KEY:
        params_2014["key"] = CENSUS_API_KEY
    response_2014 = requests.get(url_2014, params=params_2014)
    response_2014.raise_for_status()
    data_2014 = response_2014.json()
    df_2014 = pd.DataFrame(data_2014[1:], columns=data_2014[0])
    df_2014['YEAR'] = 2014
    all_dataframes.append(df_2014)
    print(f"Successfully fetched {len(df_2014)} rows for 2014.\n")

    # Combine all dataframes into one and save to CSV
    raw_census_df = pd.concat(all_dataframes, ignore_index=True)
    raw_census_df.to_csv(csv_path, index=False)
    print(f"Combined dataframe has {len(raw_census_df)} rows and {len(raw_census_df.columns)} columns.")

File '../data/census_2012_2014_sba.csv' exists. Skipping API data load.


C:\Users\Jack\AppData\Local\Temp\ipykernel_34212\1249505987.py:7: DtypeWarning: Columns (1,2,4,6,8,10,16,17,18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_census_df = pd.read_csv(csv_path)


Loaded dataframe from '../data/census_2012_2014_sba.csv' with 2205543 rows and 23 columns.


## Load Census data for 2002 and 2007 (CSV)

In [7]:
import os
import requests
# Replace with your actual Dropbox direct download links for each file
sb2002_path = os.path.join('../data', 'census_2002_sba.csv')
sb2002_url = 'https://www.dropbox.com/scl/fi/4k2w0hvhl5clwsefl4lll/census_2002_sba.csv?rlkey=kvwjfifle9ov24d4vfsjhvuyi&st=2u8t41kq&dl=1'
sb2007_path = os.path.join('../data', 'census_2007_sba.csv')
sb2007_url = 'https://www.dropbox.com/scl/fi/6lw0l0gmzddcnvn0v6ebc/census_2007_sba.csv?rlkey=coiqvmv8i91z0lbhynk1aornr&st=iahz4mnu&dl=1'

if not os.path.exists(sb2002_path):
    print("Downloading census_2002_sba.csv ...")
    response = requests.get(sb2002_url)
    response.raise_for_status()
    with open(sb2002_path, 'wb') as f:
        f.write(response.content)
    print("Downloaded census_2002_sba.csv")
else:
    print("census_2002_sba.csv already exists.")

if not os.path.exists(sb2007_path):
    print("Downloading census_2007_sba.csv ...")
    response = requests.get(sb2007_url)
    response.raise_for_status()
    with open(sb2007_path, 'wb') as f:
        f.write(response.content)
    print("Downloaded census_2007_sba.csv.")
else:
    print("census_2007_sba.csv already exists.")

Downloaded census_2002_sba.csv
Downloaded census_2007_sba.csv.


In [18]:
FRED_API_KEY = "175c981a568a2ea162c4d2a03cf1209c"

series_ids = ["FEDFUNDS", "MPRIME", "CPIAUCSL", "UNRATE"]
base_url = "https://api.stlouisfed.org/fred/series/observations"

all_series_df = []

for series_id in series_ids:
    params = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "frequency": "m",
        "observation_start": "1987-01-01",
        "observation_end": "2014-12-31"
    }
    
    response = requests.get(base_url, params=params)
    response.raise_for_status()
    data = response.json()
    
    df = pd.DataFrame(data["observations"])[["date", "value"]]
    df.rename(columns={"value": series_id}, inplace=True)
    
    all_series_df.append(df)

merged_df = all_series_df[0]
for series_df in all_series_df[1:]:
    merged_df = pd.merge(merged_df, series_df, on="date", how="outer")

merged_df.to_csv("../data/fred_1987_2014.csv", index=False)
print("\nData saved to fred_1987_2014.csv")


Data saved to fred_1987_2014.csv
